# Step 1: Maven POM Migration (Layer 1)

This notebook simulates and prototypes the Maven POM migration flow on the `sonar-stash` project.

To ensure that we do not modify the original project files, we copy the codebase to an isolated working directory `working/sonar-stash` and perform all POM updates and dependency migrations there.

In [1]:
import sys
import os
import json
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root added to path: {PROJECT_ROOT}")

# Check availability of key imports
from src.tools.maven import MavenProject, MavenPomEditor, MavenRunner

print("Successfully imported tools!")
# Setup paths globally (Adjust ORIGINAL_PROJECT_PATH to your target project folder)
ORIGINAL_PROJECT_PATH = Path("freshbrew_data/sonar-stash").resolve()
PROJECT_PATH = Path("working/sonar-stash").resolve()
pom_path = PROJECT_PATH / "pom.xml"
ARTIFACTS_DIR = ORIGINAL_PROJECT_PATH / "artifacts"

review_file = ARTIFACTS_DIR / "reader_review.json"
if not review_file.exists():
    review_file = Path("dependency_focus_scopes.json").resolve()

print(f"Original Project path: {ORIGINAL_PROJECT_PATH}")
print(f"Working Project path: {PROJECT_PATH}")
print(f"Artifacts path: {ARTIFACTS_DIR}")
print(f"Working POM file path: {pom_path}")
print(f"Review report path: {review_file} (exists: {review_file.exists()})")

Project root added to path: D:\capstone_project\MYGRATE---Capstone-Project
Successfully imported tools!
Original Project path: D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\sonar-stash
Working Project path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash
Artifacts path: D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\sonar-stash\artifacts
Working POM file path: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\pom.xml
Review report path: D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\sonar-stash\artifacts\reader_review.json (exists: True)


In [2]:
# Load reports generated by prior steps
review_data = {}
selected_sol = {}

if review_file.exists():
    with open(review_file, "r", encoding="utf-8") as f:
        review_data = json.load(f)
    review_body = review_data.get("review_upgrade_candidates", {})
    selected_sol = review_body.get("selected_solution", {})
    print("Selected Solution (from reader_review.json):")
    print(json.dumps(selected_sol, indent=2))
else:
    print("reader_review.json not found!")

Selected Solution (from reader_review.json):
{
  "org.sonarsource.sonarqube:sonar-plugin-api": "9.3.0.51899",
  "com.github.cliftonlabs:json-simple": "4.0.1",
  "org.asynchttpclient:async-http-client": "3.0.2",
  "com.google.guava:guava": "20.0",
  "org.assertj:assertj-core": "3.26.3",
  "org.mockito:mockito-core": "5.15.2",
  "org.mockito:mockito-junit-jupiter": "5.15.2",
  "org.awaitility:awaitility": "4.1.1",
  "com.github.tomakehurst:wiremock": "2.27.2",
  "org.picocontainer:picocontainer": "2.14.2",
  "com.google.code.gson:gson": "2.12.0"
}


## 1. Simulation of Lớp MVN (Maven POM Migration)

First, we copy the entire codebase from `freshbrew_data/sonar-stash` into `working/sonar-stash` to isolate the migration process.

In [3]:
# Prepare the isolated working directory by copying the codebase
import shutil

if PROJECT_PATH.exists():
    print(f"Cleaning existing working directory at {PROJECT_PATH}...")
    shutil.rmtree(PROJECT_PATH)

print(f"Copying codebase from {ORIGINAL_PROJECT_PATH} to {PROJECT_PATH}...")
# Ignore temporary build and VCS folders to make copying fast
shutil.copytree(
    ORIGINAL_PROJECT_PATH,
    PROJECT_PATH,
    ignore=shutil.ignore_patterns("target", ".git", ".github", ".idea")
)
print("Codebase successfully copied to working directory! All migration tasks will be run here.")

Copying codebase from D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\sonar-stash to D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash...
Codebase successfully copied to working directory! All migration tasks will be run here.


### 1.1 POM Migration Pipeline Functions

Here we construct the POM migration flow into clear, reusable Python functions without mocking any data. The Agent (LLM) decides what is correct vs. incorrect based on the payload, generates tool calls, and applies them dynamically to `pom.xml`.

In [4]:
def build_pom_migration_payload(pom_path, selected_sol, target_java_version="17"):
    """
    Parses the current POM and recommended upgrades to build the payload for the LLM.
    """
    pom_project = MavenProject(str(pom_path))
    pom_editor = pom_project.get_pom_editor()

    current_properties = {}
    properties_node = pom_editor.root.find("m:properties", namespaces=pom_editor.namespaces)
    if properties_node is not None:
        for prop in properties_node:
            tag = prop.tag.split("}")[-1] if "}" in prop.tag else prop.tag
            current_properties[tag] = prop.text.strip() if prop.text else ""

    current_dependencies = []
    deps_elem = pom_editor.root.xpath(".//m:dependency", namespaces=pom_editor.namespaces)
    for dep in deps_elem:
        gid = dep.findtext("m:groupId", namespaces=pom_editor.namespaces)
        aid = dep.findtext("m:artifactId", namespaces=pom_editor.namespaces)
        ver = dep.findtext("m:version", namespaces=pom_editor.namespaces) or "Managed"
        current_dependencies.append(f"{gid}:{aid}:{ver}")

    payload = {
        "instruction": (
            "You are an expert Java migration agent. Analyze the Maven POM and decide on the necessary tool calls to update it.\n"
            f"Criteria for analysis:\n"
            f"- Target Java version is {target_java_version}.\n"
            f"- What is INCORRECT: Any compiler source/target property set to values less than {target_java_version} (e.g. 1.8, 8). Any dependency version that differs from the recommended upgrades mapping.\n"
            f"- What is CORRECT: Any property already set to {target_java_version}, and any dependency version already matching the recommended mapping.\n"
            "Generate a tool call list to ensure all compiler versions are updated, and all dependencies are updated to their recommended version."
        ),
        "target_java_version": target_java_version,
        "recommended_upgrades_mapping": selected_sol,
        "current_properties": current_properties,
        "current_dependencies": current_dependencies
    }
    return payload, current_properties, current_dependencies


def get_pom_migration_actions(payload, current_properties, current_dependencies, selected_sol):
    """
    Calls local Ollama LLM to generate the JSON list of POM updates.
    Falls back to a logic-based rule list if Ollama is offline.
    """
    from langchain_ollama import ChatOllama
    from langchain_core.messages import SystemMessage, HumanMessage

    # Pre-calculate fallback actions matching the logic
    fallback_actions = []
    target_java = payload["target_java_version"]
    for prop_name in ["maven.compiler.source", "maven.compiler.target", "jdk.version"]:
        if current_properties.get(prop_name) != target_java:
            fallback_actions.append({
                "action": "ensure_property",
                "property_name": prop_name,
                "version": target_java
            })
    for dep_key, target_ver in selected_sol.items():
        if ":" in dep_key:
            gid, aid = dep_key.split(":")
            for cd in current_dependencies:
                if cd.startswith(f"{gid}:{aid}:"):
                    current_ver = cd.split(":")[-1]
                    if current_ver != target_ver:
                        fallback_actions.append({
                            "action": "update_dependency",
                            "group_id": gid,
                            "artifact_id": aid,
                            "version": target_ver
                        })
                    break

    # Configure maven-jdeprscan-plugin for Java 17 migrations
    fallback_actions.append({
        "action": "add_plugin",
        "group_id": "org.apache.maven.plugins",
        "artifact_id": "maven-jdeprscan-plugin",
        "version": "3.1.2",
        "executions": [
            {
                "goals": ["jdeprscan"]
            }
        ]
    })

    ollama_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    ollama_model = os.getenv("OLLAMA_MODEL", "gemma4:31b-cloud")
    api_key = os.getenv("OLLAMA_KEY", "")

    system_instruction = (
        "You are a Java migration assistant. Your job is to output a JSON array representing the POM edit operations needed.\n"
        "Compare current properties and dependencies with recommended target upgrades. Identify which properties/dependencies are incorrect and must be updated.\n"
        "When upgrading to Java 17, you MUST also add the 'maven-jdeprscan-plugin' under <plugins> to continuously monitor deprecations.\n"
        "Each element in the array must be an object with the following schema:\n"
        "{\n"
        "  \"action\": \"ensure_property\" | \"update_dependency\" | \"add_dependency\" | \"add_plugin\",\n"
        "  \"property_name\": \"string (required only for ensure_property)\",\n"
        "  \"group_id\": \"string (required only for update_dependency/add_dependency/add_plugin)\",\n"
        "  \"artifact_id\": \"string (required only for update_dependency/add_dependency/add_plugin)\",\n"
        "  \"version\": \"string (required)\",\n"
        "  \"executions\": \"array of objects (optional, only for add_plugin, e.g. [{'goals': ['jdeprscan']}])\"\n"
        "}\n"
        "Output ONLY raw JSON code block. Do not provide explanations, notes, or additional markdown outside the JSON.\n"
        "Do not mock or add any fake dependencies."
    )

    user_message = f"Based on this migration context, generate the POM update actions:\n{json.dumps(payload, indent=2)}"

    # Initialize ChatOllama with authentication headers if OLLAMA_KEY is present
    kwargs = {
        "model": ollama_model,
        "base_url": ollama_url,
        "temperature": 0,
    }
    if api_key:
        kwargs["client_kwargs"] = {"headers": {"Authorization": f"Bearer {api_key}"}}
        kwargs["async_client_kwargs"] = {"headers": {"Authorization": f"Bearer {api_key}"}}

    print(f"Calling Ollama LLM at {ollama_url} (model: {ollama_model})...")
    try:
        llm = ChatOllama(**kwargs)
        messages = [
            SystemMessage(content=system_instruction),
            HumanMessage(content=user_message)
        ]
        response = llm.invoke(messages)
        raw_output = response.content.strip()

        print("\n--- LLM RAW RESPONSE ---")
        print(raw_output)

        import re
        if raw_output.startswith("```"):
            match = re.search(r"```(?:json)?\s*(.*?)\s*```", raw_output, re.DOTALL)
            if match:
                raw_output = match.group(1)

        actions = json.loads(raw_output)
        print(f"\nSuccessfully parsed {len(actions)} actions from Ollama!")
        return actions
    except Exception as e:
        print(f"\n[WARNING] Failed to call/parse Ollama: {e}")
        print(f"Falling back to pre-calculated actions list based on analysis: {len(fallback_actions)} actions.")
        return fallback_actions


def execute_pom_migration_actions(actions, project_path, selected_sol):
    """
    Executes the generated actions on the pom.xml of the project.
    Validates and corrects coordinates against selected_sol to prevent LLM version/groupId mix-ups.
    """
    results = []
    print("=== APPLYING MIGRATION ACTIONS ON THE POM ===")

    p_path = Path(project_path)
    root_pom = p_path / "pom.xml"
    if not root_pom.exists():
        print(f"No root pom.xml found at {root_pom}")
        return []

    project = MavenProject(str(root_pom))
    editor = project.get_pom_editor()

    for idx, action in enumerate(actions, start=1):
        act_type = action.get("action")
        gid = action.get("group_id")
        aid = action.get("artifact_id")
        version = action.get("version")
        prop_name = action.get("property_name")
        executions = action.get("executions")
        configuration = action.get("configuration")

        # Coordinate Validation & Correction to avoid LLM hallucinations (e.g. org.sonarsource.api:sonar-plugin-api:5.4.0)
        if act_type in ("update_dependency", "add_dependency") and gid and aid:
            matched_key = None
            for key in selected_sol:
                if key.endswith(f":{aid}"):
                    matched_key = key
                    break
            if matched_key:
                target_gid, target_aid = matched_key.split(":")
                target_version = selected_sol[matched_key]
                if gid != target_gid or version != target_version:
                    print(f"  [VALIDATOR] Correcting dependency: {gid}:{aid}:{version} -> {target_gid}:{target_aid}:{target_version}")
                    gid = target_gid
                    version = target_version

        status = "ok"
        try:
            if act_type == "add_dependency":
                editor.add_dependency(gid, aid, version)
            elif act_type == "update_dependency":
                if editor.dependency_exists(gid, aid):
                    def update_func(dep_elem):
                        version_elem = editor.ensure_element(dep_elem, "m:version")
                        version_elem.text = version
                    editor.update_dependency(gid, aid, update_func)
                else:
                    editor.add_dependency(gid, aid, version)
            elif act_type == "ensure_property":
                editor.ensure_property(prop_name, version)
            elif act_type == "add_plugin":
                editor.add_plugin(
                    group_id=gid,
                    artifact_id=aid,
                    version=version,
                    configuration=configuration,
                    executions=executions
                )
            else:
                status = "error"
        except Exception as e:
            status = "error"
            print(f"  [ERROR] Action {act_type} failed: {e}")

        detail = prop_name or f"{gid}:{aid}"
        print(f"  [{idx}] Action: {act_type}, target: {detail} -> status: {status}")
        results.append((action, status))
    return results


def verify_pom_state(pom_path):
    """
    Reads the updated POM file and prints the current properties and dependencies.
    """
    updated_project = MavenProject(str(pom_path))
    updated_editor = updated_project.get_pom_editor()

    updated_properties = {}
    properties_node = updated_editor.root.find("m:properties", namespaces=updated_editor.namespaces)
    if properties_node is not None:
        for prop in properties_node:
            tag = prop.tag.split("}")[-1] if "}" in prop.tag else prop.tag
            updated_properties[tag] = prop.text.strip() if prop.text else ""

    updated_dependencies = []
    deps_elem = updated_editor.root.xpath(".//m:dependency", namespaces=updated_editor.namespaces)
    for dep in deps_elem:
        gid = dep.findtext("m:groupId", namespaces=updated_editor.namespaces)
        aid = dep.findtext("m:artifactId", namespaces=updated_editor.namespaces)
        ver = dep.findtext("m:version", namespaces=updated_editor.namespaces) or "Managed"
        updated_dependencies.append(f"{gid}:{aid}:{ver}")

    print("\n=== UPDATED POM STATE ===")
    print("Properties:")
    for k, v in updated_properties.items():
        print(f"  <{k}>: {v}")
    print("\nDependencies:")
    for dep in updated_dependencies:
        print(f"  - {dep}")
    return updated_properties, updated_dependencies

In [5]:
# Run the complete POM migration pipeline dynamically using our functions
llm_payload, cur_props, cur_deps = build_pom_migration_payload(pom_path, selected_sol, "17")

print("=== LLM CONTEXT PAYLOAD ===")
print(json.dumps(llm_payload, indent=2))

actions = get_pom_migration_actions(llm_payload, cur_props, cur_deps, selected_sol)
execute_pom_migration_actions(actions, PROJECT_PATH, selected_sol)
verify_pom_state(pom_path)

=== LLM CONTEXT PAYLOAD ===
{
  "instruction": "You are an expert Java migration agent. Analyze the Maven POM and decide on the necessary tool calls to update it.\nCriteria for analysis:\n- Target Java version is 17.\n- What is INCORRECT: Any compiler source/target property set to values less than 17 (e.g. 1.8, 8). Any dependency version that differs from the recommended upgrades mapping.\n- What is CORRECT: Any property already set to 17, and any dependency version already matching the recommended mapping.\nGenerate a tool call list to ensure all compiler versions are updated, and all dependencies are updated to their recommended version.",
  "target_java_version": "17",
  "recommended_upgrades_mapping": {
    "org.sonarsource.sonarqube:sonar-plugin-api": "9.3.0.51899",
    "com.github.cliftonlabs:json-simple": "4.0.1",
    "org.asynchttpclient:async-http-client": "3.0.2",
    "com.google.guava:guava": "20.0",
    "org.assertj:assertj-core": "3.26.3",
    "org.mockito:mockito-core": "

Calling Ollama LLM at https://ollama.com (model: gemma4:31b-cloud)...

--- LLM RAW RESPONSE ---
```json
[
  {
    "action": "ensure_property",
    "property_name": "maven.compiler.source",
    "version": "17"
  },
  {
    "action": "ensure_property",
    "property_name": "maven.compiler.target",
    "version": "17"
  },
  {
    "action": "update_dependency",
    "group_id": "org.sonarsource.sonarqube",
    "artifact_id": "sonar-plugin-api",
    "version": "9.3.0.51899"
  },
  {
    "action": "update_dependency",
    "group_id": "com.github.cliftonlabs",
    "artifact_id": "json-simple",
    "version": "4.0.1"
  },
  {
    "action": "update_dependency",
    "group_id": "org.asynchttpclient",
    "artifact_id": "async-http-client",
    "version": "3.0.2"
  },
  {
    "action": "update_dependency",
    "group_id": "com.google.guava",
    "artifact_id": "guava",
    "version": "20.0"
  },
  {
    "action": "update_dependency",
    "group_id": "org.assertj",
    "artifact_id": "assertj-core

({'sonar.version': '6.7',
  'sonar.pluginName': 'Stash',
  'sonar.pluginClass': 'org.sonar.plugins.stash.StashPlugin',
  'project.reporting.outputEncoding': 'UTF-8',
  'test.sonarqube.dist.groupId': 'fixme.fixme',
  'test.sonarqube.dist.artifactId': 'sonarqube-dist',
  'test.sonarqube.dist.version': '7.6',
  'test.sonarqube.dist.outputdir': '${project.build.directory}/fixtures/sonarqube',
  'test.sonarscanner.dist.groupId': 'fixme.fixme',
  'test.sonarscanner.dist.artifactId': 'sonarscanner-dist',
  'test.sonarscanner.dist.version': '3.3.0.1492',
  'test.sonarscanner.dist.outputdir': '${project.build.directory}/fixtures/sonarscanner',
  'test.url.binaries.repo': 'https://binaries.sonarsource.com/Distribution',
  'test.plugin.archive': '${project.build.directory}/${project.artifactId}-${project.version}.jar',
  'test.sources.dir': '${project.build.directory}/fixtures/sources',
  'sonar.coverage.exclusions': 'src/main/java/org/sonar/plugins/stash/StashPlugin.java,src/main/java/org/sonar/

In [6]:
print("=== STEP 1 POM MIGRATION COMPLETION SUMMARY ===")
print(f"POM migration completed successfully in isolation!")
print(f"The working POM has been updated at: {pom_path}")

=== STEP 1 POM MIGRATION COMPLETION SUMMARY ===
POM migration completed successfully in isolation!
The working POM has been updated at: D:\capstone_project\MYGRATE---Capstone-Project\working\sonar-stash\pom.xml
